---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！

# 2.1.4 广播机制详解

本 notebook 详细介绍 PyTorch 的广播（Broadcasting）机制

In [ ]:
import torch
import numpy as np

## 1. 什么是广播机制

广播（Broadcasting）是一种在不同形状的张量之间进行运算的机制，允许在不显式复制数据的情况下对不同形状的张量进行运算。

In [ ]:
# 简单示例：标量与张量运算
a = torch.tensor([1, 2, 3])
b = 10

result = a + b
print(f"a: {a}")
print(f"b: {b}")
print(f"a + b: {result}")
print("\n解释: 标量 10 被广播到与 a 相同的形状 [10, 10, 10]")

## 2. 广播规则

两个张量可以进行广播运算需要满足以下条件：

1. **从右向左对齐维度**
2. **每个维度必须满足以下条件之一**：
   - 两个张量在该维度上大小相同
   - 其中一个张量在该维度上大小为 1
   - 其中一个张量在该维度上不存在

In [ ]:
print("广播规则示例:")
print("="*60)

# 示例1: 形状 (3, 4) 和 (4,)
a = torch.randn(3, 4)
b = torch.randn(4)
print(f"a 形状: {a.shape}")
print(f"b 形状: {b.shape}")
print(f"a + b 形状: {(a + b).shape}")
print("可以广播: b 的形状 (4,) 扩展为 (1, 4)，然后广播到 (3, 4)")

# 示例2: 形状 (3, 1) 和 (1, 4)
print("\n" + "="*60)
a = torch.randn(3, 1)
b = torch.randn(1, 4)
print(f"a 形状: {a.shape}")
print(f"b 形状: {b.shape}")
print(f"a + b 形状: {(a + b).shape}")
print("可以广播: a 的第2维扩展到 4，b 的第1维扩展到 3")

# 示例3: 形状 (5, 3, 1) 和 (1, 4)
print("\n" + "="*60)
a = torch.randn(5, 3, 1)
b = torch.randn(1, 4)
print(f"a 形状: {a.shape}")
print(f"b 形状: {b.shape}")
print(f"a + b 形状: {(a + b).shape}")
print("可以广播: 从右向左对齐，b 扩展为 (1, 1, 4)，然后广播到 (5, 3, 4)")

## 3. 广播过程详解

In [ ]:
# 详细演示广播过程
print("广播过程详解:")
print("="*60)

# 示例: (3, 1) + (1, 4)
a = torch.tensor([[1], [2], [3]])  # shape: (3, 1)
b = torch.tensor([[10, 20, 30, 40]])  # shape: (1, 4)

print(f"a 形状: {a.shape}")
print(f"a:\n{a}")
print(f"\nb 形状: {b.shape}")
print(f"b:\n{b}")

result = a + b
print(f"\n结果形状: {result.shape}")
print(f"结果:\n{result}")

print("\n广播过程:")
print("1. a 的第2维 (1) 扩展到 4: [[1,1,1,1], [2,2,2,2], [3,3,3,3]]")
print("2. b 的第1维 (1) 扩展到 3: [[10,20,30,40], [10,20,30,40], [10,20,30,40]]")
print("3. 逐元素相加")

## 4. 常见广播模式

### 4.1 向量与矩阵

In [ ]:
# 模式1: 每行加同一个向量
matrix = torch.tensor([[1, 2, 3],
                       [4, 5, 6],
                       [7, 8, 9]])
vector = torch.tensor([10, 20, 30])

result = matrix + vector
print("模式1: 每行加同一个向量")
print(f"matrix 形状: {matrix.shape}")
print(f"vector 形状: {vector.shape}")
print(f"\nmatrix:\n{matrix}")
print(f"\nvector: {vector}")
print(f"\n结果:\n{result}")

# 模式2: 每列加同一个向量
print("\n" + "="*60)
print("模式2: 每列加同一个向量")
column_vector = torch.tensor([[10], [20], [30]])
result2 = matrix + column_vector
print(f"matrix 形状: {matrix.shape}")
print(f"column_vector 形状: {column_vector.shape}")
print(f"\nmatrix:\n{matrix}")
print(f"\ncolumn_vector:\n{column_vector}")
print(f"\n结果:\n{result2}")

### 4.2 批处理操作

In [ ]:
# 批处理示例: 对每个样本应用相同的偏置
batch_data = torch.randn(32, 10)  # 32个样本，每个10个特征
bias = torch.randn(10)  # 偏置向量

result = batch_data + bias
print(f"batch_data 形状: {batch_data.shape}")
print(f"bias 形状: {bias.shape}")
print(f"结果形状: {result.shape}")
print("\n每个样本都加上了相同的偏置向量")

# 图像批处理示例
print("\n" + "="*60)
images = torch.randn(32, 3, 224, 224)  # [batch, channels, height, width]
mean = torch.tensor([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)

normalized = (images - mean) / std
print(f"images 形状: {images.shape}")
print(f"mean 形状: {mean.shape}")
print(f"std 形状: {std.shape}")
print(f"归一化后形状: {normalized.shape}")
print("\n每个通道使用不同的均值和标准差进行归一化")

## 5. 不能广播的情况

In [ ]:
print("不能广播的情况:")
print("="*60)

# 情况1: 维度不兼容
a = torch.randn(3, 4)
b = torch.randn(3, 5)
print(f"a 形状: {a.shape}")
print(f"b 形状: {b.shape}")

try:
    result = a + b
except RuntimeError as e:
    print(f"错误: 第2维不兼容 (4 vs 5)")

# 情况2: 两个非1的维度不相同
print("\n" + "="*60)
a = torch.randn(2, 3)
b = torch.randn(4, 3)
print(f"a 形状: {a.shape}")
print(f"b 形状: {b.shape}")

try:
    result = a + b
except RuntimeError as e:
    print(f"错误: 第1维不兼容 (2 vs 4)")

## 6. 使用 unsqueeze 辅助广播

In [ ]:
# 有时需要手动添加维度以满足广播条件
a = torch.tensor([1, 2, 3])
b = torch.tensor([10, 20, 30])

print(f"a 形状: {a.shape}")
print(f"b 形状: {b.shape}")

# 直接相加是逐元素的
print(f"\na + b = {a + b} (逐元素相加)")

# 如果想要外积形式的加法
a_col = a.unsqueeze(1)  # shape: (3, 1)
b_row = b.unsqueeze(0)  # shape: (1, 3)

result = a_col + b_row
print(f"\na_col 形状: {a_col.shape}")
print(f"b_row 形状: {b_row.shape}")
print(f"\n外积形式的加法:\n{result}")
print("\n解释: 每个 a[i] 与每个 b[j] 相加")

## 7. 实际应用示例

### 7.1 神经网络的批归一化

In [ ]:
# 批归一化示例
features = torch.randn(32, 64, 28, 28)  # [batch, channels, height, width]

# 计算每个通道的均值和标准差
mean = features.mean(dim=[0, 2, 3], keepdim=True)  # shape: (1, 64, 1, 1)
std = features.std(dim=[0, 2, 3], keepdim=True)    # shape: (1, 64, 1, 1)

# 归一化（利用广播）
normalized = (features - mean) / (std + 1e-5)

print(f"features 形状: {features.shape}")
print(f"mean 形状: {mean.shape}")
print(f"std 形状: {std.shape}")
print(f"归一化后形状: {normalized.shape}")
print("\n每个通道使用自己的均值和标准差进行归一化")

### 7.2 注意力机制中的广播

In [ ]:
# 注意力权重应用示例
values = torch.randn(32, 10, 64)  # [batch, seq_len, dim]
attention_weights = torch.randn(32, 10, 1)  # [batch, seq_len, 1]

# 对每个位置的特征加权（利用广播）
weighted_values = values * attention_weights

print(f"values 形状: {values.shape}")
print(f"attention_weights 形状: {attention_weights.shape}")
print(f"加权后形状: {weighted_values.shape}")
print("\n每个位置的64维特征都乘以对应的注意力权重")

## 8. 性能优势

广播机制的优势：
1. **内存效率**: 不需要显式复制数据
2. **计算效率**: 底层优化的向量化操作
3. **代码简洁**: 避免显式的循环

In [ ]:
import time

# 对比广播和显式循环的性能
matrix = torch.randn(1000, 1000)
vector = torch.randn(1000)

# 方法1: 使用广播
start = time.time()
result1 = matrix + vector
time1 = time.time() - start

# 方法2: 显式循环（不推荐）
start = time.time()
result2 = matrix.clone()
for i in range(matrix.shape[0]):
    result2[i] = matrix[i] + vector
time2 = time.time() - start

print(f"广播时间: {time1:.6f} 秒")
print(f"循环时间: {time2:.6f} 秒")
print(f"加速比: {time2/time1:.2f}x")
print(f"结果相同: {torch.allclose(result1, result2)}")

## 9. 总结

### 广播规则
1. 从右向左对齐维度
2. 每个维度大小相同，或其中一个为 1，或其中一个不存在

### 常见模式
- 标量与张量
- 向量与矩阵
- 批处理操作
- 特征归一化

### 关键要点
1. 广播不会实际复制数据，内存高效
2. 使用 `unsqueeze` 可以添加维度以满足广播条件
3. 使用 `keepdim=True` 可以保持维度便于广播
4. 广播比显式循环更快、更简洁
5. 理解广播对于编写高效的深度学习代码很重要

---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！